# Prédiction de l'approbation d'un prêt bancaire

Ce notebook combine le contexte théorique et une démonstration pratique (code) pour construire, entraîner et évaluer des modèles de classification supervisée permettant de prédire si une demande de prêt sera acceptée (Y) ou refusée (N). Le dataset suggéré est `Loan Prediction` (Kaggle).

## 1) Contexte du problème

Les banques reçoivent chaque jour de nombreuses demandes de prêt. Accorder un crédit à un client qui ne pourra pas le rembourser représente un risque financier important. L'objectif est donc de construire un modèle capable de prédire automatiquement si une demande de prêt sera acceptée ou refusée à partir des informations du client (revenu, historique de crédit, montant demandé, etc.).

## 2) Type de classification

- **Classification binaire**.
- Justification : la variable cible `Loan_Status` prend deux valeurs possibles (Yes / No) correspondant à prêt accordé ou refusé.

## 3) Variables d'entrée et variable cible

**Variables d'entrée (exemples)** :
- `Gender` : Sexe du client
- `Married` : Situation matrimoniale
- `Dependents` : Nombre de personnes à charge
- `Education` : Niveau d'éducation
- `Self_Employed` : Auto-entrepreneur ou non
- `ApplicantIncome` : Revenu du demandeur
- `CoapplicantIncome` : Revenu du co-demandeur
- `LoanAmount` : Montant du prêt demandé
- `Loan_Amount_Term` : Durée du prêt
- `Credit_History` : Historique de crédit
- `Property_Area` : Zone géographique

**Variable cible (Output)** : `Loan_Status` → `Y` (prêt accordé) / `N` (prêt refusé)

## 4) Algorithmes choisis

Ce problème est un cas de **machine learning supervisé** : on entraîne un modèle avec des exemples étiquetés (features + target). La cible est `Loan_Status`, ce qui fait de ce cas une **classification binaire**.

**A. Régression Logistique** :
- Modèle linéaire adapté aux problèmes binaires.
- Il apprend un score de probabilité pour chaque observation.
- Utilisé comme référence (baseline) car il est stable et facile à expliquer.
- Permet de comprendre l'influence de chaque variable sur la prédiction.

**B. Random Forest** :
- Algorithme d'ensemble qui combine plusieurs arbres de décision.
- Moins sensible aux données bruitées et aux relations non linéaires.
- Réduit la variance et limite le surapprentissage par agrégation.
- Intéressant dans le secteur bancaire pour sa robustesse.

**Pourquoi ces deux modèles ?**
- La régression logistique sert à vérifier un modèle simple et interprétable.
- Le Random Forest sert à tester un modèle plus complexe et performant.
- Cela permet de comparer un modèle linéaire à un modèle non linéaire, ce que votre professeur peut demander.

**Questions attendues du professeur** :
- Pourquoi s'agit-il d'une classification binaire ?
- Quel rôle joue la variable cible ?
- Pourquoi comparer deux modèles différents ?
- Qu'est-ce qu'une matrice de confusion et pourquoi l'utiliser ?
- Quelle métrique est la plus pertinente selon le contexte métier ?

## 4.1) Vocabulaire clé et questions d'examen

- **Supervised learning** : apprentissage supervisé avec des exemples étiquetés.
- **Features** : variables d'entrée utilisées pour l'entraînement.
- **Target / label** : variable cible à prédire (`Loan_Status`).
- **Classification binaire** : deux classes possibles, ici Yes / No.
- **Dataset déséquilibré** : quand une classe domine l'autre, ce qui peut fausser l'accuracy.
- **Overfitting** : surapprentissage, le modèle mémorise trop les données d'entraînement.
- **Underfitting** : le modèle est trop simple, il ne capture pas la structure des données.
- **Généralisation** : capacité du modèle à bien prédire sur de nouvelles données.
- **Matrice de confusion** : outil d'évaluation qui montre vrais positifs, faux positifs, vrais négatifs, faux négatifs.
- **Precision** : proportion des prédictions positives correctes.
- **Recall** : proportion des vrais positifs identifiés par le modèle.
- **F1 Score** : moyenne harmonique de precision et recall.

Questions que le professeur peut poser :
- Pourquoi ce problème est-il supervisé ?
- Pourquoi s'agit-il d'une classification binaire ?
- Quel est le rôle de la target et des features ?
- Pourquoi comparer régression logistique et Random Forest ?
- Quelles métriques utilisez-vous et pourquoi ?
- Quelles sont les limites du modèle et comment l'améliorer ?


## 5) Démonstration pratique (exécutable dans Google Colab ou localement)

Les étapes : chargement → prétraitement → encodage → split → entraînement → évaluation.

In [ ]:
# Imports et configuration
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
sns.set(style='whitegrid')

In [ ]:
# Chargement des données (priorité au dataset local du projet)
candidates = [Path('data/raw/train.csv'), Path('data/raw/loan_prediction.csv'), Path('data/loan_prediction.csv')]
df = None
for p in candidates:
    if p.exists():
        df = pd.read_csv(p)
        print(f'Chargé depuis: {p}')
        break

if df is None:
    try:
        from google.colab import files  # type: ignore
        print('Aucun dataset local trouvé. Utilisez lupload en Colab.')
        uploaded = files.upload()
        name = next(iter(uploaded.keys()))
        df = pd.read_csv(name)
    except Exception:
        print('Aucun dataset trouvé. Création dun petit jeu de démonstration.')
        df = pd.DataFrame([
            {'Gender':'Male','Married':'Yes','Dependents':'0','Education':'Graduate','Self_Employed':'No','ApplicantIncome':5000,'CoapplicantIncome':0,'LoanAmount':100,'Loan_Amount_Term':360,'Credit_History':1,'Property_Area':'Urban','Loan_Status':'Y'},
            {'Gender':'Female','Married':'No','Dependents':'1','Education':'Not Graduate','Self_Employed':'No','ApplicantIncome':3000,'CoapplicantIncome':1500,'LoanAmount':85,'Loan_Amount_Term':360,'Credit_History':0,'Property_Area':'Rural','Loan_Status':'N'},
            {'Gender':'Male','Married':'Yes','Dependents':'2','Education':'Graduate','Self_Employed':'No','ApplicantIncome':4200,'CoapplicantIncome':0,'LoanAmount':120,'Loan_Amount_Term':360,'Credit_History':1,'Property_Area':'Semiurban','Loan_Status':'Y'}
        ])

df.head()

In [ ]:
# Aperçu rapide
print('Taille du dataset :', df.shape)
display(df.head())

### Prétraitement
- Imputation simple : mode pour catégoriques, median pour numériques.
- Encodage : `LabelEncoder` pour les colonnes de type object.

In [ ]:
# Prétraitement simple
df_clean = df.copy()
if 'Loan_Status' in df_clean.columns and df_clean['Loan_Status'].dtype == object:
    df_clean['Loan_Status'] = df_clean['Loan_Status'].map({'Y':1, 'N':0})
for col in df_clean.columns:
    if df_clean[col].isnull().any():
        if df_clean[col].dtype == 'O':
            df_clean[col].fillna(df_clean[col].mode().iloc[0], inplace=True)
        else:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
le = LabelEncoder()
cat_cols = [c for c in df_clean.columns if df_clean[c].dtype == 'O' and c != 'Loan_Status']
for c in cat_cols:
    df_clean[c] = le.fit_transform(df_clean[c].astype(str))
df_clean.head()

In [ ]:
# Séparation X / y et split
X = df_clean.drop('Loan_Status', axis=1)
y = df_clean['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if y.nunique()>1 else None)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## Modèle 1 : Régression Logistique

Entraînement, prédiction, matrice de confusion et métriques.

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
cm_lr = confusion_matrix(y_test, y_pred_lr)
acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_lr = f1_score(y_test, y_pred_lr, zero_division=0)
print('Logistic Regression')
print('Accuracy:', round(acc_lr,4))
print('Precision:', round(prec_lr,4))
print('Recall:', round(recall_lr,4))
print('F1 Score:', round(f1_lr,4))
plt.figure(figsize=(5,4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Logistic Regression')
plt.xlabel('Prédiction')
plt.ylabel('Réel')
plt.show()

## Modèle 2 : Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred_rf)
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
print('Random Forest')
print('Accuracy:', round(acc_rf,4))
print('Precision:', round(prec_rf,4))
print('Recall:', round(recall_rf,4))
print('F1 Score:', round(f1_rf,4))
plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Matrice de confusion - Random Forest')
plt.xlabel('Prédiction')
plt.ylabel('Réel')
plt.show()

In [ ]:
# Comparaison synthétique des métriques
results = pd.DataFrame([
    {'model':'LogisticRegression','accuracy':acc_lr,'precision':prec_lr,'recall':recall_lr,'f1':f1_lr},
    {'model':'RandomForest','accuracy':acc_rf,'precision':prec_rf,'recall':recall_rf,'f1':f1_rf}
])
results.set_index('model', inplace=True)
display(results)
results.plot(kind='bar', figsize=(8,4))
plt.title('Comparaison des modèles')
plt.ylim(0,1)
plt.show()

## 6. Exemple de résultats (attendus)

| Modèle | Accuracy | Precision | Recall | F1 Score |
|---|---:|---:|---:|---:|
| Régression Logistique | ~0.80 | ~0.80 | ~0.88 | ~0.84 |
| Random Forest | ~0.83 | ~0.83 | ~0.90 | ~0.86 |

*Les valeurs exactes dépendent du dataset et de l'exécution.*

## 7. Conclusion et pistes d'amélioration

**Quel algorithme ai-je utilisé et pourquoi ?**
- J'ai entraîné deux modèles : **Régression Logistique** et **Random Forest**.
- La régression logistique a été utilisée car elle est adaptée aux problèmes de classification binaire et donne un résultat interprétable. C'est une excellente baseline.
- Le Random Forest a été choisi pour tester un modèle plus complexe, capable de capturer des relations non linéaires entre les variables.

**L'ai-je comparé à d'autres algorithmes ?**
- Oui, la comparaison se fait dans ce notebook entre ces deux algorithmes.
- Les métriques utilisées sont **Accuracy**, **Precision**, **Recall** et **F1 Score**, avec les matrices de confusion pour visualiser les erreurs.
- Cette comparaison répond à une question d'examen classique : comparer un modèle linéaire simple à un modèle d'ensemble robuste.

**Limites des modèles utilisés**
- La régression logistique peut être limitée si les relations entre features et target sont non linéaires.
- Le Random Forest est moins interprétable ; il est difficile de comprendre précisément pourquoi le modèle prend une décision.
- Les deux modèles dépendent fortement du prétraitement : valeurs manquantes, encodage, normalisation, sélection de variables.
- Le notebook n'intègre pas de validation croisée ni de réglage fin d'hyperparamètres, ce qui peut réduire la capacité de généralisation.
- Si les classes sont déséquilibrées, l'accuracy seule ne suffit pas pour juger un bon modèle.

**Points d'amélioration**
- Ajouter une validation croisée (`CrossValidation`) pour mieux estimer la performance réelle.
- Optimiser les hyperparamètres avec `GridSearchCV` ou `RandomizedSearchCV`.
- Effectuer un **feature engineering** et une sélection de variables.
- Tester d'autres modèles : **SVM**, **KNN**, **XGBoost**, **LightGBM**, ou un arbre de décision.
- Évaluer le modèle avec des courbes **ROC / AUC** et des métriques spécifiques à la classe positive.
- Améliorer l'interprétabilité avec des méthodes de type **SHAP** ou **LIME**.

> Ce notebook présente des réponses complètes aux questions que le professeur peut poser : choix de modèle, comparaison, limites, vocabulaire de ML supervisé et axes d'amélioration.
